# CareSpace 3D · CPU 重現驗證 v1

操作指南：docs/colab-start-here.md。先只執行第一個程式碼區塊；看到
29 passed 後停止，確認結果再繼續。不要一開始選「全部執行」。
雲端來源 ZIP：我的雲端硬碟 / CareSpace3D / care-space-3d-source.zip
Colab 讀取路徑：/content/drive/MyDrive/CareSpace3D/care-space-3d-source.zip
重現輸出目錄：/content/drive/MyDrive/CareSpace3D/work-v01/

This notebook installs and invokes the same project core. It contains no separate
geometry/model implementation. It has not been executed in this development session.
The verified adapter uses CPU float32; no particular Colab GPU is assumed.

Place the provided `care-space-3d-source.zip` in `MyDrive/CareSpace3D/`. This is source
code, not a dataset collection task. The scripts acquire the exact legal research
subset (8.67 MB) and one checkpoint (1.34 GB) themselves. ReplicaCAD's pinned LICENSE
is CC BY-NC 4.0 despite the official page's CC BY 4.0 label; local research only.
Assets, models, per-frame predictions and results persist in the Drive work folder.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from zipfile import ZipFile
import subprocess, sys, shutil
SOURCE = Path('/content/drive/MyDrive/CareSpace3D/care-space-3d-source.zip')
PROJECT = Path('/content/drive/MyDrive/CareSpace3D/work-v01')
PROJECT.mkdir(parents=True, exist_ok=True)
with ZipFile(SOURCE) as z:
    for member in z.infolist():
        destination = (PROJECT / member.filename).resolve()
        if not destination.is_relative_to(PROJECT.resolve()):
            raise ValueError('Unexpected archive path')
    z.extractall(PROJECT)
ENV = Path('/content/carespace-venv')
subprocess.run([sys.executable, '-m', 'venv', str(ENV)], check=True)
PYTHON = str(ENV / 'bin/python')
subprocess.run([PYTHON, '-m', 'pip', 'install', '-r', str(PROJECT/'requirements-cpu.lock')], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', '--no-deps', '-e', str(PROJECT)], check=True)
subprocess.run([PYTHON, '-m', 'pytest', '-q', str(PROJECT/'tests')], cwd=PROJECT, check=True)


In [ ]:
# Downloads resume; existing matching predictions are reused.
for script in ['fetch_assets.py', 'fetch_model.py', 'run_study.py', 'run_learning.py', 'build_report.py', 'build_depth_audit.py', 'run_abstention.py']:
    subprocess.run([PYTHON, str(PROJECT/'scripts'/script)], cwd=PROJECT, check=True)
print((PROJECT/'docs/results.md').read_text())


In [ ]:
# Optional notebook-local viewer; uses the same static frontend.
if shutil.which('npm'):
    subprocess.run(['npm', 'ci'], cwd=PROJECT, check=True)
    subprocess.run(['node', '--test', *map(str, sorted((PROJECT/'tests').glob('browser-*.test.mjs')))], cwd=PROJECT, check=True)
    subprocess.run(['node', str(PROJECT/'scripts/build_diagnostics.mjs')], cwd=PROJECT, check=True)
    server = subprocess.Popen([PYTHON, '-m', 'http.server', '8840', '--bind', '127.0.0.1'], cwd=PROJECT)
    from google.colab import output
    output.serve_kernel_port_as_iframe(8840, path='/viewer/', height=800)
else:
    print('Node/npm unavailable; analysis artifacts are saved. Use the local viewer commands in README.md.')
